## What bowling stats most predict match wins?
Economy rate, wickets per match, dot ball %. Find correlation with team win rate. 

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

# Apply cleaning
matches['city'].fillna('Unknown', inplace=True)
matches['player_of_match'].fillna('Unknown', inplace=True)
matches['winner'].fillna('No Result', inplace=True)
matches['season'] = matches['season'].replace({
    '2007/08': '2008', '2009/10': '2010', '2020/21': '2020'
}).astype(int)

name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant':'Rising Pune Supergiants'
}
for col in ['team1','team2','winner','toss_winner']:
    matches[col] = matches[col].replace(name_map)

deliveries = deliveries[deliveries['inning'] <= 2]  # remove super overs

C:\Users\hp\AppData\Local\Temp\ipykernel_13088\1044635566.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matches['city'].fillna('Unknown', inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_13088\1044635566.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example,

In [5]:
# ── Q2 — Bowling & Wins (04_analysis_bowling.ipynb) ──
# Step 1: bowling stats
bowling_stats = deliveries.groupby('bowler').agg(
    balls_bowled=('total_runs', 'count'),
    runs_conceded=('total_runs', 'sum'),
    wickets=('is_wicket', 'sum'),
    matches_bowled=('match_id', 'nunique')
).reset_index()

bowling_stats['economy'] = (bowling_stats['runs_conceded'] / bowling_stats['balls_bowled']) * 6
bowling_stats['wickets_per_match'] = bowling_stats['wickets'] / bowling_stats['matches_bowled']

# Step 2: team win rates
wins = matches[matches['winner'] != 'No Result']
win_counts = wins['winner'].value_counts().reset_index()
win_counts.columns = ['team', 'wins']
total_matches = pd.concat([matches['team1'], matches['team2']]).value_counts().reset_index()
total_matches.columns = ['team', 'total']
team_winrate = win_counts.merge(total_matches, on='team')
team_winrate['win_rate'] = team_winrate['wins'] / team_winrate['total']
print(team_winrate.sort_values('win_rate', ascending=False))

                           team  wins  total  win_rate
8                Gujarat Titans    28     45  0.622222
1           Chennai Super Kings   138    238  0.579832
0                Mumbai Indians   144    261  0.551724
9          Lucknow Super Giants    24     44  0.545455
2         Kolkata Knight Riders   131    251  0.521912
6              Rajasthan Royals   112    221  0.506787
10      Rising Pune Supergiants    15     30  0.500000
3   Royal Challengers Bengaluru   123    255  0.482353
5                Delhi Capitals   115    252  0.456349
7                  Punjab Kings   112    246  0.455285
4           Sunrisers Hyderabad   117    257  0.455253
11                Gujarat Lions    13     30  0.433333
13         Kochi Tuskers Kerala     6     14  0.428571
12                Pune Warriors    12     46  0.260870


Team Win Rates

Gujarat Titans have the highest win rate (62.2%) of any franchise despite being the newest, a strong signal that modern squad analytics beats brand value and star power.
Chennai Super Kings and Mumbai Indians have maintained 55%+ win rates across 238 and 261 matches respectively, the only two franchises to be consistently elite at scale.
Royal Challengers Bengaluru's 48% win rate despite three of cricket's biggest stars (Kohli, de Villiers, Maxwell) shows that individual brilliance without team balance doesn't win trophies.
Pune Warriors' 26% rate remains the worst in IPL history, a cautionary tale for poor franchise management.